<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week5/lesson16/Lesson16_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 16 — Exercises**
## Probability Rules: Events, Conditioning and Detection

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 5 — Probability\
**Lesson:** Lesson 16

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Events and the axioms | 15 |
| 2 | Joint, marginal and conditional tables | 15 |
| 3 | Independence, conditional independence and total probability | 15 |
| 4 | Pooling and the mix of trials | 15 |
| 5 | Confusion matrices and base rates | 15 |
| 6 | Thresholds and trade-offs | 10 |
| 7 | Real data: a second mouse, a second detector | 15 |
| **Total** | | **100** |
| Bonus | Many detectors, many false alarms | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

print("Setup complete.")

---
## Exercise 1 — Events and the Axioms (15 points)

A simulated session of 600 trials. On each trial a grating of contrast 0, 0.25, 0.5 or 1 is shown (0 = no grating), the mouse may respond ("go"), and a go response may be fast (reaction time below 300 ms). Every array below has shape `(600,)`, one entry per trial.

In [ ]:
# 1.1 (5 points): Build the events as Boolean arrays of shape (600,):
#   S = grating shown (contrast > 0);  G = go response;  F = fast go response (rt_ms < 300; rt_ms is NaN on no-go trials).
# Then store in p1 (a dict) the empirical probabilities with keys
#   'S', 'G', 'F', 'S and G', 'S or G', 'G but not F', 'neither S nor G'.

rng1 = np.random.default_rng(1)
contrast1 = rng1.choice([0.0, 0.25, 0.5, 1.0], size=600, p=[0.3, 0.2, 0.2, 0.3])        # (600,) contrast, 0-1
go1 = rng1.random(600) < np.where(contrast1 > 0, 0.4 + 0.5 * contrast1, 0.25)              # (600,) True on go trials
rt_ms1 = np.where(go1, rng1.gamma(6.0, 55.0, size=600), np.nan)                          # (600,) reaction time, ms; NaN if no go

# YOUR CODE HERE
S1 = G1 = F1 = p1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.1
assert p1 is not None and S1 is not None, "Set S1, G1, F1 and p1."
assert S1.shape == G1.shape == F1.shape == (600,) and S1.dtype == G1.dtype == F1.dtype == bool, "Events must be Boolean arrays of shape (600,)."
assert not np.any(F1 & ~G1), "A fast go response is a go response: F must be a subset of G."
assert np.isclose(p1['S or G'], p1['S'] + p1['G'] - p1['S and G']), "Inclusion-exclusion must hold exactly."
assert np.isclose(p1['neither S nor G'], 1 - p1['S or G']) and np.isclose(p1['G but not F'], p1['G'] - p1['F']), "Check complement and difference."
assert abs(p1['S'] - 0.7) < 0.06 and p1['F'] <= p1['G'], "P(S) should be near 0.7, and P(F) <= P(G)."
print("Self-check passed.")

In [ ]:
# 1.2 (5 points): Some proposed probabilities are impossible. Using only the axioms, P(A and B) must satisfy
#     max(0, P(A) + P(B) - 1) <= P(A and B) <= min(P(A), P(B)).
# (a) Write consistent(pA, pB, pAB) returning True if all three numbers lie in [0, 1] and satisfy these bounds (allow 1e-12 slack).
# (b) Apply it to the claims below and store the list of True/False results in ok12.
# (c) For each claim, store in implied_union12 the implied P(A or B) by inclusion-exclusion (even for impossible claims).

claims12 = [(0.3, 0.5, 0.4),      # "30% of trials have a grating on the left, 50% on the right, 40% on both"
            (0.7, 0.6, 0.2),      # "70% go, 60% grating, 20% go with grating"
            (0.7, 0.6, 0.45),
            (0.2, 0.9, 0.2)]

# YOUR CODE HERE
ok12 = implied_union12 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert ok12 is not None and implied_union12 is not None, "Set ok12 and implied_union12."
assert [bool(x) for x in ok12] == [False, False, True, True], "Exactly the last two claims are possible."
assert np.allclose(implied_union12, [0.4, 1.1, 0.85, 0.9]), "Use P(A or B) = P(A) + P(B) - P(A and B)."
assert consistent(0.5, 0.5, 0.0) and not consistent(0.5, 0.5, -0.1) and not consistent(1.2, 0.5, 0.5), "Check the edge cases."
print("Self-check passed.")

**1.3 (5 points, written response).** (a) Derive the two bounds of 1.2 from the axioms (one line each), and say what goes wrong in claims 1 and 2. (b) In 1.1, $F \subseteq G$. Which rule guarantees $P(F) \le P(G)$, and how would you prove it from the axioms? (c) A student writes "fast responses and slow responses are mutually exclusive, so they are independent". Correct the statement.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Joint, Marginal and Conditional Tables (15 points)

A simulated orientation-tuned neuron: on each of 1,200 trials one grating orientation is shown, and the neuron's response in the window is classified as `'low'`, `'medium'` or `'high'`. The experimenter's protocol also lists orientation 180°, which was never actually shown.

In [ ]:
# 2.1 (6 points): Write joint_probs(x, y, x_levels, y_levels) returning the joint probability table, shape (len(x_levels), len(y_levels)),
# rows indexing x and columns indexing y. Raise ValueError if x and y are not aligned 1-D arrays or contain unlisted values.
# Compute P2 = joint table of (orientation, response) with the levels below, and the marginals
#   P_ori2 (shape (5,)) and P_resp2 (shape (3,)).

rng2 = np.random.default_rng(2)
ori_levels = [0, 45, 90, 135, 180]                                         # degrees; 180 listed but never shown
resp_levels = ['low', 'medium', 'high']
ori2 = rng2.choice([0, 45, 90, 135], size=1200)                             # (1200,) orientation shown, degrees
p_high = {0: 0.05, 45: 0.25, 90: 0.7, 135: 0.25}                            # the neuron prefers 90 degrees
p_med = {0: 0.25, 45: 0.45, 90: 0.25, 135: 0.45}
u2 = rng2.random(1200)
ph2 = np.array([p_high[o] for o in ori2]); pm2 = np.array([p_med[o] for o in ori2])
resp2 = np.where(u2 < ph2, 'high', np.where(u2 < ph2 + pm2, 'medium', 'low'))  # (1200,) response class

# YOUR CODE HERE
P2 = P_ori2 = P_resp2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1
assert P2 is not None and P_ori2 is not None and P_resp2 is not None, "Set P2, P_ori2 and P_resp2."
assert P2.shape == (5, 3) and np.isclose(P2.sum(), 1) and np.all(P2 >= 0), "A (5, 3) table of probabilities summing to 1."
assert np.allclose(P2[4], 0) and np.allclose(P_ori2, P2.sum(axis=1)) and np.allclose(P_resp2, P2.sum(axis=0)), "Marginals are row and column sums."
try:
    joint_probs(np.array([0, 45]), np.array(['low']), ori_levels, resp_levels); raise AssertionError("misaligned arrays should raise ValueError")
except ValueError:
    pass
try:
    joint_probs(np.array([30]), np.array(['low']), ori_levels, resp_levels); raise AssertionError("an unlisted level should raise ValueError")
except ValueError:
    pass
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Compute both conditional tables from P2:
#   resp_given_ori2[i, j] = P(response j | orientation i)  (each defined row sums to 1; the 180-degree row is undefined -> NaN)
#   ori_given_resp2[i, j] = P(orientation i | response j)  (each column sums to 1)
# Do not let NumPy emit a division warning. Then verify the product rule P(x, y) = P(y | x) P(x) wherever P(x) > 0,
# storing the Boolean result in product_rule_ok2.

# YOUR CODE HERE
resp_given_ori2 = ori_given_resp2 = product_rule_ok2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert resp_given_ori2 is not None and ori_given_resp2 is not None and product_rule_ok2 is not None, "Set all three results."
assert resp_given_ori2.shape == (5, 3) and np.all(np.isnan(resp_given_ori2[4])), "The never-shown orientation has an undefined (NaN) row."
assert np.allclose(np.nansum(resp_given_ori2[:4], axis=1), 1), "Each defined row of P(response | orientation) sums to 1."
assert np.allclose(ori_given_resp2.sum(axis=0), 1), "Each column of P(orientation | response) sums to 1."
assert product_rule_ok2 is True, "The product rule should hold."
assert abs(resp_given_ori2[2, 2] - 0.7) < 0.07 and ori_given_resp2[2, 2] > 0.5, "P(high | 90) should be near 0.7."
print("Self-check passed.")

**2.3 (4 points, written response).** (a) Which table is an **encoding** description of the neuron and which a **decoding** one? (b) $P(\text{high} \mid 90°)$ is about 0.7, but $P(90° \mid \text{high})$ comes out somewhat lower. Explain why the two numbers differ, and describe a change to the *experiment* (not the neuron) that would change $P(90° \mid \text{high})$ but leave $P(\text{high} \mid 90°)$ unchanged.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Independence, Conditional Independence and Total Probability (15 points)

Two neurons, A and B, are recorded over 4,000 trials. The animal is either **engaged** (60% of trials) or **disengaged**. Given the state, the neurons spike in the response window independently, with probabilities

| state | $P(A \mid \text{state})$ | $P(B \mid \text{state})$ |
|---|---|---|
| engaged | 0.7 | 0.6 |
| disengaged | 0.2 | 0.3 |

In [ ]:
# 3.1 (5 points): (a) Using the law of total probability and conditional independence, compute the EXACT values
#   pA3, pB3, pAB3 = P(A), P(B), P(A and B), and gap_exact3 = P(A and B) - P(A) P(B).
# (b) From the simulated data below, compute the estimated pooled gap (gap_pooled3) and the estimated gaps within each state,
#   gap_engaged3 and gap_disengaged3 (each: P(A and B | state) - P(A | state) P(B | state)).

rng3 = np.random.default_rng(3)
engaged3 = rng3.random(4000) < 0.6                                           # (4000,) behavioural state
A3 = rng3.random(4000) < np.where(engaged3, 0.7, 0.2)                        # (4000,) neuron A spikes in the window
B3 = rng3.random(4000) < np.where(engaged3, 0.6, 0.3)                        # (4000,) neuron B spikes (independent of A given the state)

# YOUR CODE HERE
pA3 = pB3 = pAB3 = gap_exact3 = gap_pooled3 = gap_engaged3 = gap_disengaged3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert all(v is not None for v in (pA3, pB3, pAB3, gap_exact3, gap_pooled3, gap_engaged3, gap_disengaged3)), "Set all seven values."
assert np.isclose(pA3, 0.5) and np.isclose(pB3, 0.48) and np.isclose(pAB3, 0.276) and np.isclose(gap_exact3, 0.036), "Check the exact values."
assert abs(gap_pooled3 - 0.036) < 0.015, "The pooled estimated gap should be near the exact 0.036."
assert abs(gap_engaged3) < 0.02 and abs(gap_disengaged3) < 0.02, "Within each state the gaps should be near zero."
print("Self-check passed.")

In [ ]:
# 3.2 (5 points): Is a pooled gap of this size distinguishable from chance? For n = 4000 and for n = 100 trials, simulate
# 2000 sessions in which A and B are truly INDEPENDENT with P(A) = 0.5 and P(B) = 0.48 (use rng32 below), compute the estimated gap
# in each session, and store the 95th percentile of |gap| in null95_3 = {4000: ..., 100: ...}.
# Then set dependent_4000 and dependent_100 to True if a gap equal to gap_exact3 would exceed that chance level.

rng32 = np.random.default_rng(32)

# YOUR CODE HERE
null95_3 = dependent_4000 = dependent_100 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert null95_3 is not None and dependent_4000 is not None and dependent_100 is not None, "Set null95_3 and both flags."
assert 0.005 < null95_3[4000] < 0.011 and 0.035 < null95_3[100] < 0.065, "The chance level is about 0.008 at n = 4000 and about 0.05 at n = 100."
assert dependent_4000 is True and dependent_100 is False, "The same true dependence is detectable with 4000 trials but not with 100."
print("Self-check passed.")

**3.3 (5 points, written response).** (a) The two neurons share no synapse, yet they are dependent. Where does the dependence come from, and why does it vanish within each state? (b) In the language of systems neuroscience, how would you describe the pooled co-firing (a "signal" or "state-driven" correlation) versus the within-state co-firing (a "noise" correlation)? (c) Give a short example in which two events are independent overall but dependent given a third (hint: think of two independent causes of one observed effect, and condition on the effect).

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Pooling and the Mix of Trials (15 points)

Two labs train mice on the same task, which has easy and hard trials. Each lab reports its mice's correct-response counts:

| | easy: correct / trials | hard: correct / trials |
|---|---|---|
| Lab 1 | 270 / 300 | 420 / 700 |
| Lab 2 | 760 / 800 | 130 / 200 |

In [ ]:
# 4.1 (6 points): Compute, for each lab,
#   within4 (shape (2, 2): rows lab 1, lab 2; columns easy, hard) = P(correct | difficulty),
#   mix4    (shape (2, 2))  = P(difficulty) in that lab's trials,
#   overall4 (shape (2,))   = P(correct) pooled over all of that lab's trials,
# and check that overall4 equals the law-of-total-probability mixture of within4 with weights mix4.
# Then compute standardised4 (shape (2,)): each lab's P(correct) on a COMMON mix of 50% easy and 50% hard trials.

correct4 = np.array([[270, 420], [760, 130]])                                 # (2 labs, 2 difficulties)
trials4 = np.array([[300, 700], [800, 200]])

# YOUR CODE HERE
within4 = mix4 = overall4 = standardised4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert all(v is not None for v in (within4, mix4, overall4, standardised4)), "Set within4, mix4, overall4 and standardised4."
assert np.allclose(within4, [[0.9, 0.6], [0.95, 0.65]]) and np.allclose(mix4.sum(axis=1), 1), "Check the within-difficulty rates and mixes."
assert np.allclose(overall4, [0.69, 0.89]) and np.allclose(standardised4, [0.75, 0.80]), "Check pooled and standardised rates."
print("Self-check passed.")

In [ ]:
# 4.2 (4 points): Now consider Lab 3, whose mice score 0.85 on easy and 0.55 on hard trials (worse than Lab 1 on both).
# Find the fraction of easy trials f4 that Lab 3 would need for its POOLED rate to equal Lab 1's pooled rate (0.69).
# Solve the linear equation 0.85 f + 0.55 (1 - f) = 0.69 exactly, and check your answer. (With any larger f, Lab 3's pooled rate
# would EXCEED Lab 1's although Lab 3 is worse at both difficulties: Simpson's paradox.)

# YOUR CODE HERE
f4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert f4 is not None, "Set f4."
assert np.isclose(f4, 0.14 / 0.30) and np.isclose(0.85 * f4 + 0.55 * (1 - f4), 0.69), "Solve 0.55 + 0.30 f = 0.69."
print("Self-check passed.")

**4.3 (5 points, written response).** (a) Lab 1's pooled rate is far below Lab 2's (0.69 against 0.89), but Lab 2 is better on both difficulties only by 0.05. Use the law of total probability to explain how much of the pooled gap is due to the mix of trials rather than to the mice. (b) Which comparison would you report, and why? (c) Name one neuroscience setting where the same trap appears.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Confusion Matrices and Base Rates (15 points)

Use the course layout: **rows are the truth (present, absent), columns are the decision (yes, no)**.

In [ ]:
# 5.1 (6 points): Write
#   confusion(truth, decision) -> (2, 2) integer array [[hits, misses], [false alarms, correct rejections]]
#     (raise ValueError unless both are aligned 1-D Boolean arrays), and
#   rates(cm) -> dict with keys 'sensitivity', 'false_alarm', 'specificity', 'precision', 'accuracy', using NaN when a
#     denominator is zero (no division warnings).
# Apply them to the spike-detector output below; store the matrix in cm5 and the rates in r5.

truth5 = np.array([1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0], dtype=bool)     # (20,) a real spike in this 1 ms window?
decision5 = np.array([1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0], dtype=bool)  # (20,) detector said "spike"

# YOUR CODE HERE
cm5 = r5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert cm5 is not None and r5 is not None, "Set cm5 and r5."
assert np.array_equal(cm5, [[4, 2], [2, 12]]), "Rows: truth present/absent; columns: decision yes/no."
assert np.isclose(r5['sensitivity'], 4 / 6) and np.isclose(r5['false_alarm'], 2 / 14) and np.isclose(r5['precision'], 4 / 6) and np.isclose(r5['accuracy'], 0.8)
r_never = rates(confusion(truth5, np.zeros(20, dtype=bool)))
assert r_never['sensitivity'] == 0 and np.isnan(r_never['precision']), "A detector that never says yes has undefined precision (NaN)."
try:
    confusion(truth5.astype(int), decision5); raise AssertionError("integer arrays should raise ValueError")
except ValueError:
    pass
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): A seizure detector scans EEG in 1 s windows with sensitivity 0.95 and specificity 0.98. Seizure activity
# occupies 0.2% of windows. Using the product rule and the law of total probability (no simulation), compute:
#   prec52      = precision P(seizure | alarm)
#   fa_per_hour = expected number of false alarms per hour of recording (3600 windows)
#   spec_needed = the specificity needed (sensitivity unchanged) for precision 0.5

# YOUR CODE HERE
prec52 = fa_per_hour = spec_needed = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert all(v is not None for v in (prec52, fa_per_hour, spec_needed)), "Set prec52, fa_per_hour and spec_needed."
assert np.isclose(prec52, 0.0019 / (0.0019 + 0.02 * 0.998)), "precision = sens*pi / (sens*pi + (1 - spec)(1 - pi))"
assert np.isclose(fa_per_hour, 3600 * 0.02 * 0.998), "false alarms per window times windows per hour"
assert np.isclose(spec_needed, 1 - 0.0019 / 0.998) and spec_needed > 0.998, "Solve (1 - spec)(1 - pi) = sens * pi."
print("Self-check passed.")

**5.3 (4 points, written response).** (a) The seizure detector's accuracy is about 0.98. Why is that number nearly useless here? (b) Which two numbers would you put in the abstract of a paper about this detector, and why those? (c) Software from another lab returns the matrix `[[TN, FP], [FN, TP]]`. What goes wrong if you read it with the course layout?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Thresholds and Trade-offs (10 points)

A neuron's spike count in a 250 ms window is used to detect a tone. The decision is "yes" if the count is **at least** $t$. The first 400 trials are for choosing $t$; the last 400 are for reporting.

In [ ]:
# 6.1 (6 points): (a) For every integer threshold t in thresholds6, compute the hit rate and false-alarm rate on the TRAINING
#   trials: arrays hit6 and fa6, both of shape (len(thresholds6),). Check that both never increase with t.
# (b) Choose t_equal6 minimising (misses + false alarms) on training trials, and t_fa6 minimising (misses + 4 x false alarms)
#   (a false alarm costs four times a miss). If several thresholds tie, take the smallest.
# (c) For both thresholds, compute the TEST hit and false-alarm rates: test6 = {'equal': (hit, fa), 'fa4': (hit, fa)}.

rng6 = np.random.default_rng(6)
tone6 = rng6.random(800) < 0.4                                               # (800,) tone present
count6 = rng6.poisson(np.where(tone6, 9.0, 4.0))                             # (800,) spikes in 250 ms
train6 = np.arange(800) < 400; test6_rows = ~train6
thresholds6 = np.arange(0, 21)

# YOUR CODE HERE
hit6 = fa6 = t_equal6 = t_fa6 = test6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert all(v is not None for v in (hit6, fa6, t_equal6, t_fa6, test6)), "Set hit6, fa6, t_equal6, t_fa6 and test6."
assert hit6.shape == fa6.shape == (21,) and hit6[0] == 1 and fa6[0] == 1, "At t = 0 every trial is a 'yes'."
assert np.all(np.diff(hit6) <= 0) and np.all(np.diff(fa6) <= 0), "Both rates must fall (weakly) as t rises."
assert t_fa6 >= t_equal6, "Making false alarms costlier should not lower the threshold."
assert test6['fa4'][1] <= test6['equal'][1] + 1e-12 and test6['fa4'][0] <= test6['equal'][0] + 1e-12, "The higher threshold gives fewer of both."
print("Self-check passed.")

**6.2 (4 points, written response).** (a) Why does raising $t$ lower the hit rate and the false-alarm rate **together**? (b) Why might the training-half rates at the chosen threshold be slightly better than the test-half rates, and when would that optimism be large? (c) Why must you state whether the rule is "count $\ge t$" or "count $> t$"?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: A Second Mouse, a Second Detector (15 points)

**Data.** Session 25 of the Neuromatch Academy release of [Steinmetz et al. (2019)](https://doi.org/10.1038/s41586-019-1787-x) (mouse Radnitz, 9 January 2017), stored in the course repository as `data/steinmetz_s25_radnitz.npz`. Same task as the teaching notebook's mini-lab: gratings of contrast 0, 0.25, 0.5 or 1 on each side, wheel response +1 (turn left), −1 (turn right) or 0 (no turn). Neurons were recorded in the left hemisphere, so right-side gratings are in the field represented by the recorded VISp.

| Array | Shape | Meaning, units |
|---|---|---|
| `counts7` | (178, 1056) | spikes of each neuron, 50–250 ms after stimulus onset (rows: trials; columns: neurons) |
| `area7` | (1056,) | brain area of each neuron |
| `cl7`, `cr7` | (178,) | left and right contrast, 0–1 |
| `resp7` | (178,) | wheel response |

The provided cell loads the file from your folder, from the repository's `data` folder, or downloads it once.

In [ ]:
# PROVIDED: load session 25.
import os, urllib.request
DATA_FILE7 = 'steinmetz_s25_radnitz.npz'
for cand in [DATA_FILE7, os.path.join('..', '..', 'data', DATA_FILE7)]:
    if os.path.exists(cand):
        path7 = cand; break
else:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/peter-adepoju/math-neuro-ml/8f9e0431a21d8ac0aa016e3502735f3b8627863a/data/' + DATA_FILE7, DATA_FILE7)
    path7 = DATA_FILE7
d7 = np.load(path7)
counts7 = d7['counts'].astype(float)                                        # (178, 1056) spikes in 0.2 s
area7 = d7['brain_area']                                                    # (1056,)
cl7, cr7, resp7 = d7['contrast_left'], d7['contrast_right'], d7['response']  # (178,) each
print(counts7.shape, area7.shape, cl7.shape, cr7.shape, resp7.shape, " VISp neurons:", int(np.sum(area7 == 'VISp')))

In [ ]:
# 7.1 (5 points): Behaviour.
# (a) Treat "any grating" (left or right contrast > 0) as the signal and "any wheel turn" as the decision. Store the confusion
#     matrix (course layout) in cm71, and the mouse's hit and false-alarm rates in hit71, fa71.
# (b) Verify the law of total probability for P(turn) and store the reconstructed value in p_turn71.
# (c) Store P(R | L) and P(R | not L) in pRL71 and pRnotL71 (L, R = left/right grating present).

# YOUR CODE HERE
cm71 = hit71 = fa71 = p_turn71 = pRL71 = pRnotL71 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert all(v is not None for v in (cm71, hit71, fa71, p_turn71, pRL71, pRnotL71)), "Set all six results."
assert cm71.sum() == 178 and cm71[0].sum() == 138, "138 trials had a grating."
assert np.isclose(p_turn71, np.mean(resp7 != 0)), "The law of total probability must reproduce P(turn) exactly."
assert 0.78 < hit71 < 0.87 and 0.3 < fa71 < 0.4, "Hit rate about 0.83, false-alarm rate about 0.35."
assert abs(pRL71 - pRnotL71) < 0.1, "In this session the two sides are nearly independent."
print("Self-check passed.")

In [ ]:
# 7.2 (6 points): A VISp detector of RIGHT gratings.
# (a) score7 = total VISp spike count per trial, shape (178,).
# (b) Split trials at random with rng7 below: train7 = first half of rng7.permutation(178), test7 = the rest (Boolean masks).
# (c) Choose t7 (integer, decision: score >= t7) minimising misses + false alarms on TRAINING trials (smallest on ties).
#     Store training and test (hit, false-alarm) pairs in train_rates7 and test_rates7, and the number of training errors in train_err7.
# (d) Store the test hit rate at each right contrast in hit_by_c7 = {0.25: ..., 0.5: ..., 1.0: ...}, and check that the law of total
#     probability, with weights P(contrast | right grating) on the test trials, reproduces the overall test hit rate.

rng7 = np.random.default_rng(7)

# YOUR CODE HERE
score7 = train7 = test7 = t7 = train_rates7 = test_rates7 = train_err7 = hit_by_c7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert all(v is not None for v in (score7, train7, test7, t7, train_rates7, test_rates7, train_err7, hit_by_c7)), "Set all results."
assert score7.shape == (178,) and train7.sum() == 89 and not np.any(train7 & test7), "89 training trials, disjoint from test trials."
assert train_err7 == np.min([np.sum((score7[train7] >= t) != R7[train7]) for t in range(int(score7.max()) + 2)]), "t7 must minimise training errors."
assert test_rates7[0] < train_rates7[0] + 1e-12 or test_rates7[1] > train_rates7[1] - 1e-12, "Expect the test rates to be no better on both counts."
assert hit_by_c7[0.25] < hit_by_c7[1.0], "Low-contrast gratings should be detected less often."
print("Self-check passed.")

**7.3 (4 points, written response).** (a) Compare the training and test performance of the VISp detector. What does the difference tell you? (b) The test hit rate is quoted as one number. Using your `hit_by_c7`, explain why a lab running mostly 0.25-contrast trials would measure a different "sensitivity" for the same neurons. (c) In the teaching notebook's session, VISp seemed to respond to **left** gratings when trials were pooled, because $P(R \mid L)$ differed strongly from $P(R \mid L^{c})$. Based on 7.1(c), would you expect the same pooled artefact in this session? Explain with the law of total probability.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Many Detectors, Many False Alarms (extra credit)

You test $m = 50$ neurons for a "response", each with a detector whose false-alarm rate is $\alpha = 0.05$ on trials with no stimulus. On a no-stimulus trial:
1. If the 50 detectors are independent, show that $P(\text{at least one false alarm}) = 1 - (1 - \alpha)^m$ using the complement rule, and compare with the union bound $m\alpha$.
2. Simulate 20,000 no-stimulus trials with `np.random.default_rng(99)`, first with independent detectors, then with **dependent** detectors that share a common fluctuation: on each trial draw a shared Gaussian $z_0$ and per-neuron $z_i$, form $0.8\,z_0 + 0.6\,z_i$ (still standard normal), and raise an alarm if it exceeds the 95th percentile of the standard normal (`1.6449`). Estimate $P(\text{at least one false alarm})$ in both cases.
3. Explain the difference, and what it means for analyses that test many neurons, voxels or channels.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, a legend where appropriate, and no background grid.
- Every probability you report says what it is conditioned on, and every confusion matrix says which rows and columns are which.
- Thresholds were chosen on training trials only; reported rates come from test trials.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.